## Week 2: Attribution Logic
In this section, we analyze customer journeys using event-level data and build simple attribution models such as first-touch, last-touch, and linear attribution.

In [4]:
import pandas as pd

events = pd.read_csv(
    "events.csv",
    usecols=["customer_id", "timestamp", "campaign_id", "event_type", "traffic_source"]
)

events["timestamp"] = pd.to_datetime(events["timestamp"], errors="coerce")
events.head()

,timestamp,customer_id,event_type,traffic_source,campaign_id
0,2021-01-14 13:35:43,43812,view,Email,43.0
1,2021-12-03 21:36:50,71340,add_to_cart,Email,10.0
2,2021-12-27 08:25:15,59540,purchase,Organic,0.0
3,2022-01-22 15:06:54,3601,add_to_cart,Paid Search,30.0
4,2021-05-10 12:03:09,92735,bounce,Email,26.0


### Step 1: Keep only valid journey records
We keep only rows with usable timestamps and sort the event stream customer by customer.

In [6]:
events_clean = events.dropna(subset=["timestamp", "customer_id"]).copy()
events_clean = events_clean.sort_values(["customer_id", "timestamp"])
events_clean.head()

,timestamp,customer_id,event_type,traffic_source,campaign_id
335888,2021-06-13 12:35:27,1,view,Organic,0.0
473478,2021-07-09 19:19:25,1,add_to_cart,Paid Search,29.0
428621,2021-11-28 20:19:24,1,view,EMAIL,35.0
94193,2021-12-07 07:04:24,1,click,Paid Search,47.0
572035,2021-12-16 14:19:02,1,view,Paid Search,28.0


### Step 2: Identify conversion events
We treat purchase events as conversions and use them to connect earlier touchpoints to final outcomes.

In [8]:
purchase_events = events_clean[events_clean["event_type"].str.lower() == "purchase"].copy()
purchase_events.head()

,timestamp,customer_id,event_type,traffic_source,campaign_id
219621,2023-11-19 07:55:42,8,purchase,Paid Search,26.0
470867,2022-08-31 08:52:51,13,purchase,Paid Search,50.0
79339,2022-06-22 12:48:33,14,purchase,Organic,0.0
699187,2022-11-05 02:11:12,22,purchase,Social,40.0
544555,2022-05-06 23:28:54,23,purchase,Paid Search,42.0


### Step 3: Build pre-conversion touchpoints
For each customer who converted, we collect all events that happened before or at the purchase time.

In [10]:
journey_data = events_clean.merge(
    purchase_events[["customer_id", "timestamp"]].rename(columns={"timestamp": "purchase_time"}),
    on="customer_id",
    how="inner"
)

journey_data = journey_data[journey_data["timestamp"] <= journey_data["purchase_time"]].copy()
journey_data.head()

,timestamp,customer_id,event_type,traffic_source,campaign_id,purchase_time
0,2021-10-07 07:04:43,8,view,Social,3.0,2023-11-19 07:55:42
1,2022-03-01 09:30:22,8,view,Social,18.0,2023-11-19 07:55:42
2,2023-05-30 22:27:35,8,click,Social,24.0,2023-11-19 07:55:42
3,2023-11-02 13:19:38,8,view,Organic,0.0,2023-11-19 07:55:42
4,2023-11-19 07:55:42,8,purchase,Paid Search,26.0,2023-11-19 07:55:42


### Step 4: First-touch attribution
First-touch attribution gives full credit to the first recorded campaign touchpoint in the conversion journey.

In [12]:
first_touch = journey_data.sort_values(["customer_id", "purchase_time", "timestamp"]) \
    .groupby(["customer_id", "purchase_time"], as_index=False).first()

first_touch_summary = first_touch.groupby("campaign_id").size().reset_index(name="first_touch_conversions")
first_touch_summary = first_touch_summary.sort_values("first_touch_conversions", ascending=False)
first_touch_summary.head(10)

,campaign_id,first_touch_conversions
0,0.0,18651
44,44.0,483
31,31.0,478
17,17.0,474
45,45.0,474
4,4.0,468
18,18.0,463
29,29.0,462
19,19.0,460
32,32.0,452


### Step 5: Last-touch attribution
Last-touch attribution gives full credit to the final campaign touchpoint before conversion.

In [14]:
last_touch = journey_data.sort_values(["customer_id", "purchase_time", "timestamp"]) \
    .groupby(["customer_id", "purchase_time"], as_index=False).last()

last_touch_summary = last_touch.groupby("campaign_id").size().reset_index(name="last_touch_conversions")
last_touch_summary = last_touch_summary.sort_values("last_touch_conversions", ascending=False)
last_touch_summary.head(10)

,campaign_id,last_touch_conversions
0,0.0,8083
44,44.0,920
5,5.0,917
29,29.0,890
7,7.0,888
18,18.0,876
49,49.0,857
17,17.0,828
25,25.0,815
8,8.0,799


### Step 6: Linear attribution
Linear attribution splits one conversion equally across all touchpoints in the journey.

In [16]:
journey_counts = journey_data.groupby(["customer_id", "purchase_time"]).size().reset_index(name="num_touches")
linear_data = journey_data.merge(journey_counts, on=["customer_id", "purchase_time"], how="left")
linear_data["linear_credit"] = 1 / linear_data["num_touches"]

linear_summary = linear_data.groupby("campaign_id", as_index=False)["linear_credit"].sum()
linear_summary = linear_summary.sort_values("linear_credit", ascending=False)
linear_summary.head(10)

,campaign_id,linear_credit
0,0.0,16059.734311
44,44.0,609.883588
18,18.0,573.078135
29,29.0,567.462012
17,17.0,560.041790
7,7.0,547.304977
5,5.0,546.400468
49,49.0,546.242135
8,8.0,537.907077
25,25.0,537.308052


### Step 7: Compare attribution models
We compare how campaign performance changes under first-touch, last-touch, and linear attribution.

In [18]:
attribution_compare = first_touch_summary.merge(
    last_touch_summary, on="campaign_id", how="outer"
).merge(
    linear_summary, on="campaign_id", how="outer"
)

attribution_compare = attribution_compare.fillna(0)
attribution_compare = attribution_compare.sort_values("last_touch_conversions", ascending=False)
attribution_compare.head(15)

,campaign_id,first_touch_conversions,last_touch_conversions,linear_credit
0,0.0,18651,8083,16059.734311
44,44.0,483,920,609.883588
5,5.0,428,917,546.400468
29,29.0,462,890,567.462012
7,7.0,450,888,547.304977
18,18.0,463,876,573.078135
49,49.0,436,857,546.242135
17,17.0,474,828,560.041790
25,25.0,441,815,537.308052
8,8.0,443,799,537.907077


### Step 8: Add campaign channel names
To make the output easier to interpret, we attach channel information from the campaigns table.

In [21]:
campaigns = pd.read_csv("campaigns.csv")
campaign_lookup = campaigns[["campaign_id", "channel", "objective"]].drop_duplicates()

attribution_compare_named = attribution_compare.merge(
    campaign_lookup, on="campaign_id", how="left"
)

attribution_compare_named.head(15)

,campaign_id,first_touch_conversions,last_touch_conversions,linear_credit,channel,objective
0,0.0,18651,8083,16059.734311,NaN,NaN
1,44.0,483,920,609.883588,Affiliate,Reactivation
2,5.0,428,917,546.400468,Social,Acquisition
3,29.0,462,890,567.462012,Email,Acquisition
4,7.0,450,888,547.304977,Paid Search,Cross-sell
5,18.0,463,876,573.078135,Affiliate,Retention
6,49.0,436,857,546.242135,Paid Search,Reactivation
7,17.0,474,828,560.041790,Display,Retention
8,25.0,441,815,537.308052,Paid Search,Cross-sell
9,8.0,443,799,537.907077,Paid Search,Cross-sell


### Week 2 observations
- First-touch shows which campaigns introduced customers.
- Last-touch shows which campaigns were closest to conversion.
- Linear attribution shows which campaigns consistently appeared across the full journey.
- These results will support Week 3 KPI calculations such as efficiency and ROI analysis.

## Week 3: KPI Calculation and Performance Analysis
In this section, we calculate key business metrics such as total revenue, total orders, average order value, campaign performance, CAC, and ROAS.

### Step 1: Use cleaned transaction data
We use the cleaned transaction table because refunded orders were already excluded earlier in the notebook.

In [27]:
transactionsclean = pd.read_csv('transactions.csv')
transactionsclean['timestamp'] = pd.to_datetime(transactionsclean['timestamp'], errors='coerce')
transactionsclean.head()

,transaction_id,timestamp,customer_id,product_id,quantity,discount_applied,gross_revenue,campaign_id,refund_flag
0,1,2021-12-27 08:25:15,59540,1630.0,3,0.00,43.74,0,0
1,2,2023-06-06 21:14:26,54871,1901.0,3,0.00,174.78,21,0
2,3,2023-08-31 05:29:54,51818,1884.0,1,0.00,40.61,37,0
3,4,2022-06-26 20:33:46,18164,1114.0,2,0.15,68.76,13,0
4,5,2023-07-26 18:12:35,86915,408.0,1,0.00,14.64,4,0


### Step 2: Calculate overall business KPIs
We calculate total revenue, total transactions, unique customers, and average order value.

In [29]:
total_revenue = transactionsclean["gross_revenue"].sum()
total_transactions = transactionsclean["transaction_id"].nunique()
unique_customers = transactionsclean["customer_id"].nunique()
average_order_value = total_revenue / total_transactions

kpi_summary = pd.DataFrame({
    "Metric": ["Total Revenue", "Total Transactions", "Unique Customers", "Average Order Value"],
    "Value": [total_revenue, total_transactions, unique_customers, average_order_value]
})

kpi_summary

,Metric,Value
0,Total Revenue,8.373966e+06
1,Total Transactions,1.031270e+05
2,Unique Customers,6.403500e+04
3,Average Order Value,8.120052e+01


### Step 3: Build campaign-level revenue summary
We summarize revenue and transaction counts by campaign.

In [31]:
campaign_perf = transactionsclean.groupby("campaign_id", as_index=False).agg(
    total_revenue=("gross_revenue", "sum"),
    total_orders=("transaction_id", "nunique"),
    unique_customers=("customer_id", "nunique")
)

campaign_perf["average_order_value"] = campaign_perf["total_revenue"] / campaign_perf["total_orders"]
campaign_perf.head()

,campaign_id,total_revenue,total_orders,unique_customers,average_order_value
0,0,1690120.04,20955,18689,80.654738
1,1,82045.31,946,935,86.728658
2,2,160482.03,1978,1960,81.133483
3,3,140606.61,1758,1747,79.981007
4,4,143495.73,1851,1834,77.523355


### Step 4: Add campaign details
We enrich the campaign KPI table with channel and objective information.

In [33]:
campaign_perf = campaign_perf.merge(
    campaigns[["campaign_id", "channel", "objective"]],
    on="campaign_id",
    how="left"
)

campaign_perf.head()

,campaign_id,total_revenue,total_orders,unique_customers,average_order_value,channel,objective
0,0,1690120.04,20955,18689,80.654738,NaN,NaN
1,1,82045.31,946,935,86.728658,Paid Search,Cross-sell
2,2,160482.03,1978,1960,81.133483,Email,Retention
3,3,140606.61,1758,1747,79.981007,Email,Reactivation
4,4,143495.73,1851,1834,77.523355,Display,Reactivation


### Step 5: Estimate spend using expected uplift as a proxy
Since explicit campaign spend is not clearly available in the notebook, we create an estimated spend proxy using expected uplift for comparison metrics.

In [40]:
campaign_perf = campaign_perf.merge(
    campaigns[["campaign_id"]],
    on="campaign_id",
    how="left"
)

campaign_perf["estimated_spend"] = 10000
campaign_perf.head()

,campaign_id,total_revenue,total_orders,unique_customers,average_order_value,channel,objective,estimated_spend
0,0,1690120.04,20955,18689,80.654738,NaN,NaN,10000
1,1,82045.31,946,935,86.728658,Paid Search,Cross-sell,10000
2,2,160482.03,1978,1960,81.133483,Email,Retention,10000
3,3,140606.61,1758,1747,79.981007,Email,Reactivation,10000
4,4,143495.73,1851,1834,77.523355,Display,Reactivation,10000


### Step 6: Calculate CAC and ROAS
We calculate customer acquisition cost and return on ad spend using the estimated spend proxy.

In [42]:
campaign_perf["CAC"] = campaign_perf["estimated_spend"] / campaign_perf["unique_customers"]
campaign_perf["ROAS"] = campaign_perf["total_revenue"] / campaign_perf["estimated_spend"]

campaign_perf[["campaign_id", "channel", "objective", "total_revenue", "total_orders",
               "unique_customers", "estimated_spend", "CAC", "ROAS"]].head(10)

,campaign_id,channel,objective,total_revenue,total_orders,unique_customers,estimated_spend,CAC,ROAS
0,0,NaN,NaN,1690120.04,20955,18689,10000,0.535074,169.012004
1,1,Paid Search,Cross-sell,82045.31,946,935,10000,10.695187,8.204531
2,2,Email,Retention,160482.03,1978,1960,10000,5.102041,16.048203
3,3,Email,Reactivation,140606.61,1758,1747,10000,5.724098,14.060661
4,4,Display,Reactivation,143495.73,1851,1834,10000,5.452563,14.349573
5,5,Social,Acquisition,184244.86,2231,2211,10000,4.522840,18.424486
6,6,Affiliate,Reactivation,134758.95,1690,1682,10000,5.945303,13.475895
7,7,Paid Search,Cross-sell,171224.62,2168,2146,10000,4.659832,17.122462
8,8,Paid Search,Cross-sell,160195.13,2027,2007,10000,4.982561,16.019513
9,9,Social,Retention,154304.06,1898,1886,10000,5.302227,15.430406


### Step 7: Channel-level performance summary
We compare performance by marketing channel.

In [43]:
channel_perf = campaign_perf.groupby("channel", as_index=False).agg(
    total_revenue=("total_revenue", "sum"),
    total_orders=("total_orders", "sum"),
    unique_customers=("unique_customers", "sum"),
    estimated_spend=("estimated_spend", "sum")
)

channel_perf["CAC"] = channel_perf["estimated_spend"] / channel_perf["unique_customers"]
channel_perf["ROAS"] = channel_perf["total_revenue"] / channel_perf["estimated_spend"]

channel_perf.sort_values("total_revenue", ascending=False)

,channel,total_revenue,total_orders,unique_customers,estimated_spend,CAC,ROAS
0,Affiliate,1608144.17,19694,19510,110000,5.638134,14.619492
3,Paid Search,1533909.64,18841,18666,110000,5.893068,13.944633
2,Email,1357550.66,16741,16631,110000,6.614154,12.341370
1,Display,1211416.38,14995,14859,90000,6.056935,13.460182
4,Social,972825.47,11901,11801,80000,6.779087,12.160318


### Step 8: Identify top-performing campaigns
We rank campaigns by revenue and ROAS.

In [45]:
top_revenue_campaigns = campaign_perf.sort_values("total_revenue", ascending=False).head(10)
top_roas_campaigns = campaign_perf.sort_values("ROAS", ascending=False).head(10)

top_revenue_campaigns[["campaign_id", "channel", "objective", "total_revenue", "ROAS"]]

,campaign_id,channel,objective,total_revenue,ROAS
0,0,NaN,NaN,1690120.04,169.012004
18,18,Affiliate,Retention,187497.20,18.749720
5,5,Social,Acquisition,184244.86,18.424486
29,29,Email,Acquisition,180315.11,18.031511
44,44,Affiliate,Reactivation,175026.54,17.502654
7,7,Paid Search,Cross-sell,171224.62,17.122462
25,25,Paid Search,Cross-sell,169397.56,16.939756
17,17,Display,Retention,169313.35,16.931335
16,16,Email,Reactivation,166515.24,16.651524
49,49,Paid Search,Reactivation,165853.45,16.585345


In [47]:
top_roas_campaigns[["campaign_id", "channel", "objective", "estimated_spend", "ROAS"]]

,campaign_id,channel,objective,estimated_spend,ROAS
0,0,NaN,NaN,10000,169.012004
18,18,Affiliate,Retention,10000,18.749720
5,5,Social,Acquisition,10000,18.424486
29,29,Email,Acquisition,10000,18.031511
44,44,Affiliate,Reactivation,10000,17.502654
7,7,Paid Search,Cross-sell,10000,17.122462
25,25,Paid Search,Cross-sell,10000,16.939756
17,17,Display,Retention,10000,16.931335
16,16,Email,Reactivation,10000,16.651524
49,49,Paid Search,Reactivation,10000,16.585345


### Step 9: Monthly trend analysis
We summarize revenue by month to identify trends over time.

In [49]:
transactionsclean["month"] = transactionsclean["timestamp"].dt.to_period("M").astype(str)

monthly_revenue = transactionsclean.groupby("month", as_index=False).agg(
    total_revenue=("gross_revenue", "sum"),
    total_orders=("transaction_id", "nunique")
)

monthly_revenue.head(12)

,month,total_revenue,total_orders
0,2021-01,225536.72,2813
1,2021-02,207915.83,2473
2,2021-03,228099.36,2812
3,2021-04,225594.93,2713
4,2021-05,237667.02,2859
5,2021-06,219639.13,2668
6,2021-07,229469.28,2749
7,2021-08,228921.67,2826
8,2021-09,216423.17,2721
9,2021-10,225403.28,2762


### Step 10: Week 3 observations
- Revenue and orders help measure campaign contribution.
- CAC helps estimate customer acquisition efficiency.
- ROAS helps compare revenue return relative to estimated spend.
- Channel-level results help identify which marketing sources perform best.